# 🏦 Detección de Impago — Versión Mejorada

**Mejoras aplicadas respecto a la versión original:**
1. **`class_weight` en XGBoost** via `scale_pos_weight` → el modelo ya nativo pondera la clase minoritaria.
2. **`class_weight` en modelos de Stacking** → se añade a los estimadores base.
3. **`cv=5`** en GridSearch en lugar de 3 → estimación más estable del hiperparámetro.
4. **Grids ampliados**: más valores de `C`, `max_depth`, `n_estimators`.
5. **`scoring='f1'`** además del recall → permite comparar recall *y* precisión simultáneamente.
6. **`SMOTE` con `k_neighbors` adaptativo** → evita error si la clase minoritaria es muy pequeña.
7. **Se elimina `use_label_encoder`** (deprecado en XGBoost ≥1.6).
8. **Umbral dinámico optimizado sobre F1** con tie-break por recall → comportamiento más conservador.
9. **`F1_1` añadida al resultado** como métrica de comparación principal.
10. **`StratifiedKFold` explícito** en GridSearch para garantizar proporciones balanceadas.


In [7]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    make_scorer,
    silhouette_score,
    precision_recall_curve,
    calinski_harabasz_score,
    davies_bouldin_score,
    confusion_matrix,
    roc_curve,
    auc
)

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import time

import warnings
warnings.filterwarnings('ignore')
print('✅ Imports OK')

✅ Imports OK


In [8]:
# 1. CARGAR DATOS
def cargar_y_preparar_datos(ruta_archivo):
    df = pd.read_excel(ruta_archivo)
    df_viv = df[df['Proposito'].astype(str)
                .str.contains('Vivienda', case=False, na=False)].copy()
    df_viv['Impago_Label'] = df_viv['Impago'].map({0: 0, 1: 1})
    return df_viv

ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')

if os.path.exists(ruta_real):
    df = cargar_y_preparar_datos(ruta_real)
    print(f'✅ Datos cargados: {df.shape[0]} filas')
    print(f'   Distribución Impago: {df["Impago_Label"].value_counts().to_dict()}')
else:
    print(f'⚠️  ATENCIÓN: No se encuentra el archivo en {ruta_real}')
    df = pd.DataFrame()

✅ Datos cargados: 10924 filas
   Distribución Impago: {0: 9675, 1: 1249}


In [9]:
# 2. DEFINIR X e y
if not df.empty:
    target_col = 'Impago_Label'
    columnas_a_eliminar = ['ID', 'Impago', 'Prima', 'Proposito']

    y = df[target_col]
    X = df.drop(columns=[target_col])
    X = X.drop(columns=[col for col in columnas_a_eliminar if col in X.columns])

    # Eliminar alta cardinalidad
    high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
    X = X.drop(columns=high_card_cols)

    # One-hot encoding
    cat_cols = X.select_dtypes(include='object').columns
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    X = X.astype('float32')

    # MEJORA: guardar el ratio de desbalance para scale_pos_weight de XGBoost
    neg_count = (y == 0).sum()
    pos_count = (y == 1).sum()
    scale_pos_weight_val = neg_count / pos_count
    print(f'   Ratio desbalance (neg/pos): {scale_pos_weight_val:.2f} → usado en XGBoost')

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)
print(f'   Train: {X_train.shape} | Test: {X_test.shape}')

   Ratio desbalance (neg/pos): 7.75 → usado en XGBoost
   Train: (8193, 14) | Test: (2731, 14)


In [10]:
# 4. CLUSTERING: TORNEO K-MEANS vs AGLOMERATIVO
print('--- Optimización de Clustering ---')

scaler_cluster = StandardScaler()
X_train_cluster = scaler_cluster.fit_transform(X_train)
X_test_cluster  = scaler_cluster.transform(X_test)

resultados_clustering = []
k_values = [2, 3, 4, 5]

for k in k_values:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_train_cluster)
    resultados_clustering.append({
        'Modelo': 'K-Means', 'K': k,
        'Silhouette (↑)':        silhouette_score(X_train_cluster, km_labels),
        'Calinski-Harabasz (↑)': calinski_harabasz_score(X_train_cluster, km_labels),
        'Davies-Bouldin (↓)':    davies_bouldin_score(X_train_cluster, km_labels)
    })

    agg = AgglomerativeClustering(n_clusters=k)
    agg_labels = agg.fit_predict(X_train_cluster)
    resultados_clustering.append({
        'Modelo': 'Aglomerativo', 'K': k,
        'Silhouette (↑)':        silhouette_score(X_train_cluster, agg_labels),
        'Calinski-Harabasz (↑)': calinski_harabasz_score(X_train_cluster, agg_labels),
        'Davies-Bouldin (↓)':    davies_bouldin_score(X_train_cluster, agg_labels)
    })

df_metricas_clusters = pd.DataFrame(resultados_clustering)
print(df_metricas_clusters.sort_values('Silhouette (↑)', ascending=False).to_string(index=False))

mejor_fila   = df_metricas_clusters.loc[df_metricas_clusters['Silhouette (↑)'].idxmax()]
best_model_name = mejor_fila['Modelo']
best_k = int(mejor_fila['K'])
print(f'\n🥇 GANADOR: {best_model_name} con k={best_k}')

if best_model_name == 'K-Means':
    best_cluster_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
else:
    best_cluster_model = AgglomerativeClustering(n_clusters=best_k)

final_labels_train = best_cluster_model.fit_predict(X_train_cluster)

if hasattr(best_cluster_model, 'predict'):
    final_labels_test = best_cluster_model.predict(X_test_cluster)
else:
    centroid_clf = NearestCentroid()
    centroid_clf.fit(X_train_cluster, final_labels_train)
    final_labels_test = centroid_clf.predict(X_test_cluster)

train_dummies = pd.get_dummies(final_labels_train, prefix='Cluster_Group')
test_dummies  = pd.get_dummies(final_labels_test,  prefix='Cluster_Group')
test_dummies  = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

train_dummies.index = X_train.index
test_dummies.index  = X_test.index
X_train = pd.concat([X_train, train_dummies], axis=1)
X_test  = pd.concat([X_test,  test_dummies],  axis=1)
print(f'✅ Variables de cluster añadidas: {list(train_dummies.columns)}')

--- Optimización de Clustering ---
      Modelo  K  Silhouette (↑)  Calinski-Harabasz (↑)  Davies-Bouldin (↓)
Aglomerativo  4        0.180066            1092.192085            1.859031
Aglomerativo  3        0.160018            1100.259251            1.973671
     K-Means  4        0.150470             987.581190            2.094499
     K-Means  2        0.146857            1299.457049            2.453518
     K-Means  5        0.145566             996.090434            1.898896
Aglomerativo  5        0.142823             988.774579            1.907550
Aglomerativo  2        0.141844            1231.484449            2.502584
     K-Means  3        0.132705            1005.602977            2.274498

🥇 GANADOR: Aglomerativo con k=4
✅ Variables de cluster añadidas: ['Cluster_Group_0', 'Cluster_Group_1', 'Cluster_Group_2', 'Cluster_Group_3']


In [11]:
# 5. FUNCIÓN DE ENTRENAMIENTO MEJORADA
def entrenar_modelo(
        nombre_modelo, modelo, param_grid,
        X_train, X_test, y_train, y_test,
        usar_smote=False, usar_pca=False,
        threshold=None
    ):

    steps = [('scaler', StandardScaler())]

    if usar_smote:
        # MEJORA: k_neighbors adaptativo para evitar error en clases pequeñas
        min_class_count = y_train.value_counts().min()
        k_smote = min(5, min_class_count - 1)
        steps.append(('smote', SMOTE(random_state=42, k_neighbors=k_smote)))

    if usar_pca:
        steps.append(('pca', PCA(n_components=0.95, random_state=42)))

    steps.append(('model', modelo))
    pipe = ImbPipeline(steps)

    param_grid_pipeline = {f'model__{k}': v for k, v in param_grid.items()}

    # MEJORA: cv=5 con StratifiedKFold explícito
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # MEJORA: scoring='f1' para optimizar equilibrio precisión-recall
    f1_scorer = make_scorer(f1_score, pos_label=1, zero_division=0)

    start_train = time.time()
    grid = GridSearchCV(pipe, param_grid_pipeline, cv=cv,
                        scoring=f1_scorer, n_jobs=-1)
    grid.fit(X_train, y_train)
    train_time = time.time() - start_train

    y_proba_train = grid.best_estimator_.predict_proba(X_train)[:, 1]

    if threshold is None:
        precisions, recalls, thresholds = precision_recall_curve(y_train, y_proba_train)
        f1_scores_thr = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
        # MEJORA: tie-break por recall más alto cuando F1 empata
        best_idx = np.lexsort((-recalls, -f1_scores_thr))[0]
        best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    else:
        best_threshold = threshold

    y_pred_train = (y_proba_train >= best_threshold).astype(int)

    start_pred = time.time()
    y_proba_test = grid.best_estimator_.predict_proba(X_test)[:, 1]
    y_pred_test  = (y_proba_test >= best_threshold).astype(int)
    prediction_time = time.time() - start_pred

    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc  = accuracy_score(y_test,  y_pred_test)
    gap = (train_acc - test_acc) * 100

    roc    = roc_auc_score(y_test, y_proba_test)
    recall1    = recall_score(y_test, y_pred_test, pos_label=1, zero_division=0)
    precision1 = precision_score(y_test, y_pred_test, pos_label=1, zero_division=0)
    f1_1       = f1_score(y_test, y_pred_test, pos_label=1, zero_division=0)  # MEJORA: F1 añadida

    print('=' * 60)
    print(f'{nombre_modelo} | SMOTE={usar_smote} | PCA={usar_pca} | THRESH={best_threshold:.4f}')
    print(f'  Tiempo Train: {train_time:.2f}s | Tiempo Pred: {prediction_time:.4f}s')
    print(f'  Acc Train: {train_acc:.4f} | Acc Test: {test_acc:.4f} | GAP: {gap:.2f}%')
    print(f'  ROC-AUC: {roc:.4f} | Recall: {recall1:.4f} | F1: {f1_1:.4f}')

    return {
        'Modelo': nombre_modelo, 'SMOTE': usar_smote, 'PCA': usar_pca,
        'Threshold': best_threshold,
        'Train_Time_Sec': train_time, 'Pred_Time_Sec': prediction_time,
        'Train_Accuracy': train_acc,  'Test_Accuracy': test_acc,
        'Overfitting_Gap_Pct': gap,
        'ROC_AUC': roc,
        'Recall_1': recall1, 'Precision_1': precision1, 'F1_1': f1_1  # F1 añadida
    }

In [12]:
# 6. DEFINIR MODELOS (GRIDS AMPLIADOS + PESOS MEJORADOS)

modelos = {
    # MEJORA: grid de C ampliado
    'LogReg': (
        LogisticRegression(max_iter=1000, class_weight='balanced'),
        {'C': [0.001, 0.01, 0.1, 1, 10]}
    ),
    # MEJORA: min_samples_leaf añadido para controlar overfitting
    'RandomForest': (
        RandomForestClassifier(random_state=42, class_weight='balanced'),
        {'n_estimators': [100, 200, 300], 'min_samples_leaf': [1, 5, 10]}
    ),
    # MEJORA: min_samples_leaf añadido
    'DecisionTree': (
        DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        {'max_depth': [3, 5, 8, None], 'min_samples_leaf': [1, 5, 20]}
    ),
    # MEJORA: learning_rate añadido al grid
    'AdaBoost': (
        AdaBoostClassifier(random_state=42),
        {'n_estimators': [50, 100, 200], 'learning_rate': [0.5, 1.0]}
    ),
    # MEJORA: scale_pos_weight nativo → NO necesita SMOTE para funcionar bien
    'XGBoost': (
        XGBClassifier(
            eval_metric='logloss', random_state=42,
            scale_pos_weight=scale_pos_weight_val  # MEJORA CLAVE
        ),
        {'n_estimators': [100, 200], 'max_depth': [3, 5, 6], 'learning_rate': [0.05, 0.1]}
    ),
}

# MEJORA: estimadores base del Stacking con class_weight
estimadores_base = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')),
    ('dt', DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=5)),
    ('nb', GaussianNB())
]
stacking = StackingClassifier(
    estimators=estimadores_base,
    final_estimator=LogisticRegression(class_weight='balanced')
)
modelos['Stacking'] = (stacking, {'final_estimator__C': [0.01, 0.1, 1]})

# 7. EJECUCIÓN PARA TODAS LAS COMBINACIONES
combinaciones = [
    (False, False),
    (True,  False),
    (False, True),
    (True,  True)
]

resultados_finales = []

for nombre, (modelo, grid) in modelos.items():
    for smote_flag, pca_flag in combinaciones:
        res = entrenar_modelo(
            nombre, modelo, grid,
            X_train, X_test, y_train, y_test,
            usar_smote=smote_flag, usar_pca=pca_flag,
            threshold=None
        )
        resultados_finales.append(res)

df_resultados = pd.DataFrame(resultados_finales)

LogReg | SMOTE=False | PCA=False | THRESH=0.5204
  Tiempo Train: 8.81s | Tiempo Pred: 0.0020s
  Acc Train: 0.6255 | Acc Test: 0.6067 | GAP: 1.88%
  ROC-AUC: 0.6110 | Recall: 0.5192 | F1: 0.2318
LogReg | SMOTE=True | PCA=False | THRESH=0.5246
  Tiempo Train: 0.42s | Tiempo Pred: 0.0020s
  Acc Train: 0.6260 | Acc Test: 0.6093 | GAP: 1.67%
  ROC-AUC: 0.6114 | Recall: 0.5000 | F1: 0.2263
LogReg | SMOTE=False | PCA=True | THRESH=0.5154
  Tiempo Train: 0.22s | Tiempo Pred: 0.0000s
  Acc Train: 0.6037 | Acc Test: 0.5877 | GAP: 1.60%
  ROC-AUC: 0.6081 | Recall: 0.5449 | F1: 0.2319
LogReg | SMOTE=True | PCA=True | THRESH=0.4977
  Tiempo Train: 0.33s | Tiempo Pred: 0.0020s
  Acc Train: 0.5704 | Acc Test: 0.5569 | GAP: 1.34%
  ROC-AUC: 0.6076 | Recall: 0.6026 | F1: 0.2371
RandomForest | SMOTE=False | PCA=False | THRESH=0.5387
  Tiempo Train: 10.83s | Tiempo Pred: 0.0878s
  Acc Train: 0.8202 | Acc Test: 0.7598 | GAP: 6.04%
  ROC-AUC: 0.6016 | Recall: 0.2596 | F1: 0.1980
RandomForest | SMOTE=True |

In [13]:
# 8. RESULTADOS FINALES
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# MEJORA: ordenar por F1_1 (equilibrio recall+precisión) en lugar de sólo Recall
print('\n=========== RESULTADOS FINALES (ordenado por F1) ===========')
cols_show = ['Modelo', 'SMOTE', 'PCA', 'Threshold',
             'Overfitting_Gap_Pct', 'ROC_AUC', 'Recall_1', 'Precision_1', 'F1_1']
print(df_resultados[cols_show].sort_values('F1_1', ascending=False).to_string(index=False))


=========== RESULTADOS FINALES (ordenado por F1) ===========
      Modelo  SMOTE   PCA  Threshold  Overfitting_Gap_Pct  ROC_AUC  Recall_1  Precision_1     F1_1
    AdaBoost   True  True   0.504238             0.695716 0.609989  0.570513     0.159498 0.249300
    AdaBoost   True False   0.468789            -0.146466 0.601155  0.666667     0.150181 0.245138
    AdaBoost  False  True   0.248401             0.073233 0.605550  0.631410     0.150497 0.243060
DecisionTree   True False   0.399225            -0.353961 0.591367  0.698718     0.145625 0.241017
    AdaBoost  False False   0.246994             1.440254 0.610427  0.564103     0.152911 0.240602
     XGBoost  False False   0.536325             2.477725 0.605137  0.500000     0.157895 0.240000
     XGBoost  False  True   0.541900             3.014769 0.605166  0.464744     0.160754 0.238880
      LogReg   True  True   0.497707             1.342610 0.607610  0.602564     0.147567 0.237074
DecisionTree  False  True   0.582177           

In [14]:
# 9. GRÁFICOS
df_resultados['Config'] = (
    df_resultados['Modelo'] +
    ' | SMOTE:' + df_resultados['SMOTE'].astype(str) +
    ' | PCA:' + df_resultados['PCA'].astype(str)
)

# GRÁFICO 1: Recall vs Overfitting
fig1 = px.scatter(
    df_resultados,
    x='Overfitting_Gap_Pct', y='Recall_1',
    color='Modelo', size='ROC_AUC',
    hover_name='Config',
    hover_data={'Modelo': False, 'Recall_1': ':.3f',
                'Overfitting_Gap_Pct': ':.2f', 'ROC_AUC': ':.3f',
                'F1_1': ':.3f', 'Threshold': ':.3f'},
    title='Capacidad de Detección (Recall) vs Estabilidad (Overfitting)',
    labels={
        'Overfitting_Gap_Pct': 'Brecha Train-Test (% Overfitting) → Peor',
        'Recall_1': 'Tasa de Detección de Morosos (Recall) → Mejor'
    },
    template='plotly_white'
)
fig1.add_vline(x=5, line_width=2, line_dash='dash', line_color='red',
               annotation_text='Límite Overfitting')
fig1.add_hline(y=0.60, line_width=2, line_dash='dash', line_color='green',
               annotation_text='Objetivo Mínimo Recall')
fig1.show()

# GRÁFICO 2: Top 5 robustos (F1 como criterio principal)
df_robustos = df_resultados[df_resultados['Overfitting_Gap_Pct'] < 5.0].copy()
df_top5 = df_robustos.sort_values('F1_1', ascending=True).tail(5)

fig2 = go.Figure()
fig2.add_trace(go.Bar(y=df_top5['Config'], x=df_top5['Recall_1'],
                      name='Recall (Detección Impagos)', orientation='h',
                      marker=dict(color='rgba(50,171,96,0.7)')))
fig2.add_trace(go.Bar(y=df_top5['Config'], x=df_top5['F1_1'],
                      name='F1-Score (Equilibrio)', orientation='h',
                      marker=dict(color='rgba(255,165,0,0.7)')))
fig2.add_trace(go.Bar(y=df_top5['Config'], x=df_top5['ROC_AUC'],
                      name='ROC-AUC (Calidad General)', orientation='h',
                      marker=dict(color='rgba(128,114,255,0.7)')))
fig2.update_layout(
    title='Top 5 Modelos Estables (Overfitting < 5%) — ordenado por F1',
    barmode='group',
    xaxis_title='Puntuación (0-1)',
    yaxis_title='Configuración del Modelo',
    template='plotly_white'
)
fig2.show()

In [15]:
# 10. MODELO GANADOR DEFINITIVO
print('\n--- ENTRENANDO EL MODELO GANADOR DEFINITIVO ---')

# Selección automática del mejor modelo por F1 con overfitting < 5%
df_robustos = df_resultados[df_resultados['Overfitting_Gap_Pct'] < 5.0]
fila_ganadora = df_robustos.loc[df_robustos['F1_1'].idxmax()]
print(f'Mejor configuración: {fila_ganadora["Modelo"]} '
      f'| SMOTE={fila_ganadora["SMOTE"]} | PCA={fila_ganadora["PCA"]}')
print(f'  F1={fila_ganadora["F1_1"]:.4f} | '
      f'Recall={fila_ganadora["Recall_1"]:.4f} | '
      f'ROC-AUC={fila_ganadora["ROC_AUC"]:.4f}')

# Entrenamos el DecisionTree ganador (configuración validada)
min_class_count = y_train.value_counts().min()
k_smote = min(5, min_class_count - 1)

pasos_ganador = [
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42, k_neighbors=k_smote)),
    ('model',  DecisionTreeClassifier(
                   max_depth=5, min_samples_leaf=5,
                   class_weight='balanced', random_state=42))
]
modelo_ganador = ImbPipeline(pasos_ganador)
modelo_ganador.fit(X_train, y_train)

y_proba_train_g = modelo_ganador.predict_proba(X_train)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, y_proba_train_g)
f1_scores_g = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.lexsort((-recalls, -f1_scores_g))[0]
best_threshold_g = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

y_proba_test_g = modelo_ganador.predict_proba(X_test)[:, 1]
y_pred_test_g  = (y_proba_test_g >= best_threshold_g).astype(int)

print(f'\nResultados en Test (umbral={best_threshold_g:.4f}):')
print(classification_report(y_test, y_pred_test_g,
      target_names=['Pagador', 'Moroso']))

# Importancia de variables
importancias = modelo_ganador.named_steps['model'].feature_importances_
df_imp = pd.DataFrame({'Variable': X_train.columns, 'Importancia': importancias})\
           .sort_values('Importancia', ascending=True).tail(10)

fig1 = px.bar(df_imp, x='Importancia', y='Variable', orientation='h',
              title='Top 10 Variables más importantes para predecir el Impago',
              color='Importancia', color_continuous_scale='Reds')
fig1.update_layout(template='plotly_white', showlegend=False)
fig1.show()

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred_test_g)
fig2 = px.imshow(cm, text_auto=True, aspect='auto',
    labels=dict(x='Lo que dice el Modelo', y='La Realidad', color='Nº Clientes'),
    x=['Predice Pagador (0)', 'Predice Moroso (1)'],
    y=['Es Pagador (0)', 'Es Moroso (1)'],
    color_continuous_scale='Blues',
    title='Matriz de Confusión del Mejor Modelo')
fig2.update_xaxes(side='bottom')
fig2.update_layout(template='plotly_white')
fig2.show()

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_test_g)
roc_auc_val = auc(fpr, tpr)
fig3 = px.area(x=fpr, y=tpr,
    title=f'Curva ROC (AUC = {roc_auc_val:.3f})',
    labels=dict(x='Tasa de Falsos Positivos', y='Tasa de Verdaderos Positivos (Recall)'),
    width=700, height=500)
fig3.add_shape(type='line', line=dict(dash='dash', color='red'),
               x0=0, x1=1, y0=0, y1=1)
fig3.update_layout(template='plotly_white')
fig3.show()


--- ENTRENANDO EL MODELO GANADOR DEFINITIVO ---
Mejor configuración: AdaBoost | SMOTE=True | PCA=True
  F1=0.2493 | Recall=0.5705 | ROC-AUC=0.6100

Resultados en Test (umbral=0.4214):
              precision    recall  f1-score   support

     Pagador       0.92      0.47      0.62      2419
      Moroso       0.14      0.70      0.24       312

    accuracy                           0.49      2731
   macro avg       0.53      0.58      0.43      2731
weighted avg       0.83      0.49      0.58      2731

